# GPU-Accelerated Krylov Solvers in NGSolve

**NGSolve User Meeting 2026 — Natalia Tylek, TU Wien**

This notebook shows how to use the GPU Krylov solvers in NGSolve's `ngscuda` module:
- **`DevCGSolver`** — symmetric positive-definite problems 
- **`DevTFQMRSolver`** — non-symmetric problems 

> **GPU required:** In Colab, go to **Runtime → Change runtime type** and select **T4 GPU** (free) or any GPU partition available for you.

> Any NVIDIA GPU with CUDA ≥ 12.4 works — if you have access to an A100 or H100, use that instead.


In [1]:
import sys
if 'google.colab' in sys.modules:
    !pip install --upgrade --pre ngsolve anywidget -q

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/natalia342N/starting_with_ngscuda/blob/main/docs/ngsolve_meeting_tutorial.ipynb)

#### Check if GPU is available

In [2]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
if result.returncode == 0:
    print('GPU found', result.stdout.strip())
else:
    print('No GPU found')

GPU found NVIDIA H100, 95830 MiB


In [3]:
import ngsolve
from ngsolve import *
from ngsolve import la
from ngsolve.solvers import TFQMR
from time import time

try:
    from ngsolve.ngscuda import *
except:
    print("no CUDA library or device available, using replacement types on host")

print("NGSolve version:", ngsolve.__version__)

CUDA Device Query...
There is 1 CUDA device.
CUDA Device 0: NVIDIA H100, cap 9.0
Using device 0
Initializing cublas and cusparse.
NGSolve version: 6.2.2604-9-gf15d395df


[cusparse] handle created
[InitCuLinalg] cusparseSetStream bound to ngs_cuda_stream
[InitCuLinalg] callback wired, registering creators...


---

## Part 1: Symmetric problem — Poisson equation

$$-\Delta u + u = f \quad \text{in } \Omega, \qquad u = 0 \quad \text{on } \partial\Omega$$

The stiffness matrix is **symmetric positive definite** - example case for Conjugate Gradient.

This extends the setup from
[5.5.1 Solving the Poisson Equation on devices](https://docu.ngsolve.org/latest/i-tutorials/unit-5.5-cuda/poisson_cuda.html)
with the new **CUDA graph** execution mode.

In [4]:
mesh = Mesh(unit_square.GenerateMesh(maxh=0.1))
for l in range(5): mesh.Refine()

fes = H1(mesh, order=2, dirichlet='.*')
u, v = fes.TnT()

with TaskManager():
    a = BilinearForm(grad(u)*grad(v)*dx + u*v*dx).Assemble()
    f = LinearForm(x*v*dx).Assemble()

gfu = GridFunction(fes)
jac = a.mat.CreateSmoother(fes.FreeDofs())

print(f'ndof = {fes.ndof}')

ndof = 460033


#### Solve on CPU for reference timing and solution 

In [5]:
with TaskManager():
    cpu_solver = CGSolver(a.mat, jac, maxiter=2000)
    gfu.vec.data = cpu_solver * f.vec   # warmup 1
    gfu.vec.data = cpu_solver * f.vec   # warmup 2
    ts = time()
    for _ in range(5): gfu.vec.data = cpu_solver * f.vec
    cpu_ms = (time() - ts) / 5 * 1000

ref_vec = gfu.vec.CreateVector()
ref_vec.data = gfu.vec

print(f'CPU CG:  {cpu_ms:.0f} ms')
print(f'  |sol| = {Norm(gfu.vec):.6e}')
print(f'  steps = {cpu_solver.GetSteps()}')

CPU CG:  27265 ms
  |sol| = 6.691534e+00
  steps = 1671


In [6]:
adev   = a.mat.CreateDeviceMatrix()
jacdev = jac.CreateDeviceMatrix()
fdev   = f.vec.CreateDeviceVector(copy=True)

inv = CGSolver(adev, jacdev, maxiter=2000)
gfu.vec.data = (inv * fdev).Evaluate()   # warm-up
ts = time()
for _ in range(5): gfu.vec.data = (inv * fdev).Evaluate()
gpu_std_ms = (time() - ts) / 5 * 1000

print(f'GPU CG (no graph):  {gpu_std_ms:.0f} ms')
print(f'  diff    = {Norm(gfu.vec - ref_vec):.1e}')
print(f'  speedup = {cpu_ms/gpu_std_ms:.2f}x')

GPU CG (no graph):  194 ms
  diff    = 4.6e-12
  speedup = 140.44x


### GPU acceleration 

In [7]:
gpu_solver = DevCGSolver(
    mat=adev, pre=jacdev,
    adev_raw=adev, cdev_raw=jacdev,
    maxsteps=2000, printrates=False
)

gfu.vec.data = (gpu_solver * fdev).Evaluate()  
ts = time()
for _ in range(5): gfu.vec.data = (gpu_solver * fdev).Evaluate()
gpu_ms = (time() - ts) / 5 * 1000

print(f'GPU CG (WHILE graph):  {gpu_ms:.0f} ms')
print(f'  diff    = {Norm(gfu.vec - ref_vec):.1e}')
print(f'  speedup = {cpu_ms/gpu_ms:.2f}x')

[CudaGraph] captured nodes: 17
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 17
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 17


[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 17
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 17


GPU CG (WHILE graph):  154 ms
  diff    = 1.3e-09
  speedup = 177.33x


[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 17
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful


> **Note:** `DevCGSolver` automatically selects the best available GPU execution mode:
> - **WHILE graph** (CUDA ≥ 12.4) — entire convergence loop on device, zero per-iteration host–device sync
> - **per-iteration graph** — fallback for older CUDA
> - **no-graph** — set `NO_CUDA_GRAPH=1` to disable graph capture
>
> No configuration required from the user.


### Scaling problem size

In [8]:
print(f"{'ndof':>8}  {'CPU (ms)':>10}  {'Python dev (ms)':>16}  {'WHILE graph (ms)':>17}  {'speedup':>8}")
print('-' * 68)

for maxh in [0.05, 0.03, 0.01, 0.007, 0.005]:
    m = Mesh(unit_square.GenerateMesh(maxh=maxh))
    fs = H1(m, order=2, dirichlet='.*')
    uu, vv = fs.TnT()
    with TaskManager():
        aa = BilinearForm(grad(uu)*grad(vv)*dx + uu*vv*dx).Assemble()
        ff = LinearForm(x*vv*dx).Assemble()
    jj = aa.mat.CreateSmoother(fs.FreeDofs())
    gf = GridFunction(fs)

    with TaskManager():
        cs = CGSolver(aa.mat, jj, maxsteps=2000, precision=1e-12)
        gf.vec.data = cs * ff.vec
        ts = time()
        for _ in range(3): gf.vec.data = cs * ff.vec
        c_ms = (time() - ts) / 3 * 1000

    ad = aa.mat.CreateDeviceMatrix()
    jd = jj.CreateDeviceMatrix()
    fd = ff.vec.CreateDeviceVector(copy=True)

    ng = CGSolver(ad, jd, maxsteps=2000, precision=1e-12)
    gf.vec.data = (ng * fd).Evaluate()
    ts = time()
    for _ in range(3): gf.vec.data = (ng * fd).Evaluate()
    ng_ms = (time() - ts) / 3 * 1000

    gs = DevCGSolver(mat=ad, pre=jd, adev_raw=ad, cdev_raw=jd,
                     maxsteps=2000, precision=1e-12, printrates=False)
    gf.vec.data = (gs * fd).Evaluate()
    ts = time()
    for _ in range(3): gf.vec.data = (gs * fd).Evaluate()
    g_ms = (time() - ts) / 3 * 1000

    print(f'{fs.ndof:>8}  {c_ms:>10.0f}  {ng_ms:>16.0f}  {g_ms:>17.0f}  {ng_ms/g_ms:>7.2f}x')

    ndof    CPU (ms)   Python dev (ms)   WHILE graph (ms)   speedup
--------------------------------------------------------------------


[CudaGraph] captured nodes: 17
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 17
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 17
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 17
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful


    1969         176                 7                  3     2.12x


[CudaGraph] captured nodes: 17
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 17
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 17
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 17
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful


    5233         311                11                  5     2.15x


...

   46749        1353                36                 17     2.15x


[CudaGraph] captured nodes: 17
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 17
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 17
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 17
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful


   95017        2069                52                 26     1.97x


  185857        5086                83                 49     1.68x


[CudaGraph] captured nodes: 17
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 17
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 17
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 17
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful


<details>
<summary><b>H100 reference results — Part 1 (Musica, June 2026)</b></summary>

Single run, ndof = 460,033:

| Solver | Time (ms) | Speedup vs CPU |
|---|---:|---:|
| CPU CG | ~26,000 | — |
| Python CGSolver (device matrices) | 191 | **~130×** |
| DevCGSolver (WHILE graph) | 154 | **~160×** |

Scaling (Python CGSolver vs DevCGSolver WHILE graph):

| ndof | Python CG (ms) | WHILE graph (ms) | speedup |
|---:|---:|---:|---:|
| 7,329 | 13 | 8 | **1.71×** |
| 28,993 | 26 | 16 | **1.62×** |
| 115,329 | 60 | 42 | 1.43× |
| 460,033 | 190 | 159 | 1.19× |
| 1,837,569 | 616 | 532 | 1.16× |

</details>

---

## Part 2: Non-symmetric problem — 3D convection

The same DG convection operator used in the NGSolve CUDA benchmark:
a transport-dominated 3D problem on an unstructured mesh with upwind flux.
The stiffness matrix is **non-symmetric** — CG does not apply.

**TFQMR** (Transpose-Free QMR) handles non-symmetric systems without storing a
growing Krylov basis — this is the relevant solver for **Navier–Stokes** linearizations.

In [9]:
# 3D DG convection — same setup as the CUDA benchmark
mesh2 = Mesh(unit_cube.GenerateMesh(maxh=0.05))
fes2  = L2(mesh2, order=2, dgjumps=True)
u2, v2 = fes2.TnT()

wind = CF((1, 0.2, 0.3))
n    = specialcf.normal(mesh2.dim)
dS   = dx(element_boundary=True)

with TaskManager():
    a2 = BilinearForm(fes2)
    a2 += -20*u2*v2*dx
    a2 += u2*wind*grad(v2)*dx
    a2 += -(wind*n)*IfPos(wind*n, u2, u2.Other(bnd=0))*v2*dS
    a2.Assemble()

blocks = fes2.CreateSmoothingBlocks(blocktype='element')
pre2   = a2.mat.CreateBlockSmoother(blocks)   # block element smoother
pre2p  = Projector(fes2.FreeDofs(), True)
f2     = LinearForm(1*v2*dx).Assemble()
gfu2   = GridFunction(fes2)

print(f'ndof = {fes2.ndof}')

OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
To avoid this warning, please rebuild your copy of OpenBLAS with a larger NUM_THREADS setting
OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
To avoid this warning, please rebuild your copy of OpenBLAS with a larger NUM_THREADS setting
OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
OpenBLAS warning: precompiled NUM_THREADS exceeded, adding auxiliary array for thread metadata.
To avoid this warning, please rebuild your copy of OpenBLAS with a larger NUM_THREADS setting
OpenBLAS warning: precompiled NUM_THREADS exce

BLAS : Bad memory unallocation! :  562  0x7fb72839a000
BLAS : Bad memory unallocation! :  562  0x7fb72039a000
BLAS : Bad memory unallocation! :  562  0x7fbaf039a000
BLAS : Bad memory unallocation! :  562  0x7fbaa039a000
BLAS : Bad memory unallocation! :  562  0x7fbaa839a000
BLAS : Bad memory unallocation! :  562  0x7fba3039a000
BLAS : Bad memory unallocation! :  562  0x7fba2839a000
BLAS : Bad memory unallocation! :  562  0x7fb95839a000
BLAS : Bad memory unallocation! :  562  0x7fbae039a000
BLAS : Bad memory unallocation! :  562  0x7fbac839a000
BLAS : Bad memory unallocation! :  562  0x7fba8039a000
BLAS : Bad memory unallocation! :  562  0x7fb9e039a000
BLAS : Bad memory unallocation! :  562  0x7fba4839a000
BLAS : Bad memory unallocation! :  562  0x7fbad039a000
BLAS : Bad memory unallocation! :  562  0x7fba5039a000
BLAS : Bad memory unallocation! :  562  0x7fbad839a000
BLAS : Bad memory unallocation! :  562  0x7fba8839a000
BLAS : Bad memory unallocation! :  562  0x7fbab039a000
BLAS : Bad

In [10]:
# CPU TFQMR reference
with TaskManager():
    gfu2.vec.data = TFQMR(mat=pre2@a2.mat, pre=pre2p,
                           rhs=(pre2*f2.vec).Evaluate(),
                           maxsteps=400, printrates=False, tol=1e-12)
    ts = time()
    for _ in range(3):
        gfu2.vec.data = TFQMR(mat=pre2@a2.mat, pre=pre2p,
                               rhs=(pre2*f2.vec).Evaluate(),
                               maxsteps=400, printrates=False, tol=1e-12)
    cpu2_ms = (time() - ts) / 3 * 1000

ref_norm2 = Norm(gfu2.vec)
print(f'CPU TFQMR: {cpu2_ms:.0f} ms   |sol| = {ref_norm2:.6e}')

CPU TFQMR: 3561 ms   |sol| = 8.791473e+00


In [11]:
# move data to GPU — reused by DevTFQMRSolver below
fdev2    = f2.vec.CreateDeviceVector(copy=True)
adev2    = a2.mat.CreateDeviceMatrix()
pdev2    = pre2.CreateDeviceMatrix()
pdev2p   = pre2p.CreateDeviceMatrix()
rhs_dev2 = (pdev2 * fdev2).Evaluate()
pa_dev2  = pdev2 @ adev2

# Python-level TFQMR with GPU data (per-iteration host sync)
gfu2.vec.data = TFQMR(mat=pa_dev2, pre=pdev2p, rhs=rhs_dev2,
                       maxsteps=400, printrates=False, tol=1e-12)   # warm-up
ts = time()
for _ in range(3):
    gfu2.vec.data = TFQMR(mat=pa_dev2, pre=pdev2p, rhs=rhs_dev2,
                           maxsteps=400, printrates=False, tol=1e-12)
gpu2_std_ms = (time() - ts) / 3 * 1000

print(f'GPU TFQMR (no graph):  {gpu2_std_ms:.0f} ms')
print(f'  diff    = {abs(Norm(gfu2.vec) - ref_norm2):.1e}')
print(f'  speedup = {cpu2_ms/gpu2_std_ms:.2f}x')

GPU TFQMR (no graph):  31 ms
  diff    = 1.8e-15
  speedup = 116.22x


### GPU acceleration

In [12]:
# GPU DevTFQMRSolver (WHILE graph)
tfqmr_solver = DevTFQMRSolver(
    mat=pre2 @ a2.mat, pre=pre2p,        # CPU operators (required by binding)
    adev_raw=pa_dev2, cdev_raw=pdev2p,   # GPU operators (used for computation)
    maxsteps=400, printrates=False, precision=1e-12
)

tfqmr_solver.Mult(rhs_dev2, gfu2.vec)   # warm-up
ts = time()
for _ in range(3): tfqmr_solver.Mult(rhs_dev2, gfu2.vec)
gpu2_ms = (time() - ts) / 3 * 1000

err2 = abs(Norm(gfu2.vec) - ref_norm2)
print(f'GPU TFQMR (WHILE graph):  {gpu2_ms:.0f} ms')
print(f'  diff    = {err2:.1e}')
print(f'  speedup = {cpu2_ms/gpu2_ms:.1f}x')

tion! :  562  0x7fb85039a000
BLAS : Bad memory unallocation! :  562  0x7fb99839a000
BLAS : Bad memory unallocation! :  562  0x7fb8c839a000
BLAS : Bad memory unallocation! :  562  0x7fbab839a000
BLAS : Bad memory unallocation! :  562  0x7fb93839a000
BLAS : Bad memory unallocation! :  562  0x7fb8c039a000
BLAS : Bad memory unallocation! :  562  0x7fb8b839a000
BLAS : Bad memory unallocation! :  562  0x7fb84839a000
BLAS : Bad memory unallocation! :  562  0x7fb9a039a000
BLAS : Bad memory unallocation! :  562  0x7fba6039a000
BLAS : Bad memory unallocation! :  562  0x7fb81039a000
BLAS : Bad memory unallocation! :  562  0x7fb82839a000
BLAS : Bad memory unallocation! :  562  0x7fb90839a000
BLAS : Bad memory unallocation! :  562  0x7fb7f839a000
BLAS : Bad memory unallocation! :  562  0x7fb77039a000
BLAS : Bad memory unallocation! :  562  0x7fb83039a000
BLAS : Bad memory unallocation! :  562  0x7fb75039a000
BLAS : Bad memory unallocation! :  562  0x7fb7a039a000
BLAS : Bad memory unallocation! :  5

GPU TFQMR (WHILE graph):  19 ms
  diff    = 5.3e-15
  speedup = 188.0x


[CudaGraph] captured nodes: 47
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 47
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 47
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 47
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful


### Scaling problem size

In [13]:
print(f"{'ndof':>8}  {'CPU (ms)':>10}  {'no graph (ms)':>14}  {'WHILE graph (ms)':>17}  {'speedup':>8}")
print('-' * 66)

for maxh in [0.2, 0.1, 0.07, 0.05]:
    m2 = Mesh(unit_cube.GenerateMesh(maxh=maxh))
    fs2 = L2(m2, order=2, dgjumps=True)
    uu2, vv2 = fs2.TnT()
    wind2 = CF((1, 0.2, 0.3))
    n2    = specialcf.normal(m2.dim)
    dS2   = dx(element_boundary=True)
    with TaskManager():
        aa2 = BilinearForm(fs2)
        aa2 += -20*uu2*vv2*dx
        aa2 += uu2*wind2*grad(vv2)*dx
        aa2 += -(wind2*n2)*IfPos(wind2*n2, uu2, uu2.Other(bnd=0))*vv2*dS2
        aa2.Assemble()
    blks2 = fs2.CreateSmoothingBlocks(blocktype='element')
    pp2   = aa2.mat.CreateBlockSmoother(blks2)
    pp2p  = Projector(fs2.FreeDofs(), True)
    ff2   = LinearForm(1*vv2*dx).Assemble()
    gg2   = GridFunction(fs2)

    # CPU
    with TaskManager():
        gg2.vec.data = TFQMR(mat=pp2@aa2.mat, pre=pp2p,
                              rhs=(pp2*ff2.vec).Evaluate(),
                              maxsteps=400, printrates=False, tol=1e-12)
        ts = time()
        for _ in range(3):
            gg2.vec.data = TFQMR(mat=pp2@aa2.mat, pre=pp2p,
                                  rhs=(pp2*ff2.vec).Evaluate(),
                                  maxsteps=400, printrates=False, tol=1e-12)
        c2_ms = (time() - ts) / 3 * 1000

    # GPU device objects
    fd2  = ff2.vec.CreateDeviceVector(copy=True)
    ad2  = aa2.mat.CreateDeviceMatrix()
    pd2  = pp2.CreateDeviceMatrix()
    pd2p = pp2p.CreateDeviceMatrix()
    rd2  = (pd2 * fd2).Evaluate()
    pad2 = pd2 @ ad2

    # No-graph GPU TFQMR
    gg2.vec.data = TFQMR(mat=pad2, pre=pd2p, rhs=rd2,
                          maxsteps=400, printrates=False, tol=1e-12)
    ts = time()
    for _ in range(3):
        gg2.vec.data = TFQMR(mat=pad2, pre=pd2p, rhs=rd2,
                              maxsteps=400, printrates=False, tol=1e-12)
    ng2_ms = (time() - ts) / 3 * 1000

    # DevTFQMRSolver (WHILE graph)
    gs2 = DevTFQMRSolver(mat=pp2@aa2.mat, pre=pp2p,
                          adev_raw=pad2, cdev_raw=pd2p,
                          maxsteps=400, printrates=False, precision=1e-12)
    gs2.Mult(rd2, gg2.vec)
    ts = time()
    for _ in range(3): gs2.Mult(rd2, gg2.vec)
    g2_ms = (time() - ts) / 3 * 1000

    print(f'{fs2.ndof:>8}  {c2_ms:>10.0f}  {ng2_ms:>14.0f}  {g2_ms:>17.0f}  {ng2_ms/g2_ms:>7.2f}x')

    ndof    CPU (ms)   no graph (ms)   WHILE graph (ms)   speedup
------------------------------------------------------------------


[CudaGraph] captured nodes: 47
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 47
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 47
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 47
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful


    6520         190               5                  2     2.72x


[CudaGraph] captured nodes: 47
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 47
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 47
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 47
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful


   47810         287               9                  4     2.43x


[CudaGraph] captured nodes: 47
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 47
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 47
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 47
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful


  149650        1219              16                  9     1.89x


  340370        4111              31                 19     1.62x


[CudaGraph] captured nodes: 47
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 47
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 47
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful
[CudaGraph] captured nodes: 47
[CudaWhileGraph] body graph nodes: 2
[CudaWhileGraph] Build successful


<details>
<summary><b>H100 reference results — Part 2 (Musica, June 2026)</b></summary>

Single run, ndof = 340,370:

| Solver | Time (ms) | Speedup vs CPU |
|---|---:|---:|
| CPU TFQMR | 3,148 | — |
| Python TFQMR (device matrices) | 29 | **108×** |
| DevTFQMRSolver (WHILE graph) | 19 | **165×** |

Scaling (Python TFQMR vs DevTFQMRSolver WHILE graph):

| ndof | Python TFQMR (ms) | WHILE graph (ms) | speedup |
|---:|---:|---:|---:|
| 6,520 | 5 | 2 | **2.78×** |
| 47,810 | 8 | 4 | **2.36×** |
| 149,650 | 15 | 8 | 1.94× |
| 340,370 | 29 | 19 | 1.54× |

</details>

---

## Summary

| Solver | Problem type | Typical use case |
|---|---|---|
| `DevCGSolver` | Symmetric SPD | Poisson, elasticity |
| `DevTFQMRSolver` | Non-symmetric | Convection-diffusion, Navier–Stokes |

Both solvers require no CUDA knowledge, produce numerically identical results to CPU
counterparts, and automatically select the best GPU execution strategy.

**Minimal changes to an existing NGSolve script:**

```python
from ngsolve.ngscuda import *

adev = a.mat.CreateDeviceMatrix()   # ← add
jdev = jac.CreateDeviceMatrix()     # ← add

solver = DevCGSolver(mat=adev, pre=jdev, adev_raw=adev, cdev_raw=jdev,
                     maxsteps=2000)
gfu.vec.data = (solver * f.vec.CreateDeviceVector(copy=True)).Evaluate()
```